In [8]:
from datetime import date

tickets = [
    {"id": "T-101", "team": "infrastructure", "status": "open",     "priority": "high",
     "created": date(2025, 11, 3),
     "text": "Kubernetes pod keeps crashing with OOMKilled — memory limits on the ML inference container are set too low for the model it loads at runtime."},

    {"id": "T-102", "team": "infrastructure", "status": "open",     "priority": "high",
     "created": date(2025, 11, 8),
     "text": "Nginx ingress returning 502 after rotating TLS certificate. Chain is valid per openssl verify but the backend handshake fails immediately."},

    {"id": "T-103", "team": "infrastructure", "status": "resolved", "priority": "medium",
     "created": date(2025, 10, 14),
     "text": "Terraform state file locked in S3 — a team member force-applied a plan without releasing the DynamoDB lock first."},

    {"id": "T-104", "team": "infrastructure", "status": "open",     "priority": "medium",
     "created": date(2025, 11, 12),
     "text": "Prometheus high-cardinality metric explosion — per-request labels in series names causing memory pressure on the Thanos compactor."},

    {"id": "T-105", "team": "infrastructure", "status": "resolved", "priority": "low",
     "created": date(2025, 9, 28),
     "text": "Worker node disk pressure — build cache written to ephemeral storage is never cleaned between CI jobs, filling the volume over a weekend."},

    {"id": "T-201", "team": "backend", "status": "open",     "priority": "high",
     "created": date(2025, 11, 5),
     "text": "OAuth2 token refresh fails intermittently — race condition in the token cache where two threads both detect expiry and attempt refresh simultaneously."},

    {"id": "T-202", "team": "backend", "status": "open",     "priority": "high",
     "created": date(2025, 11, 9),
     "text": "Database connection pool exhausted under load — pool capped at 20 connections but the API handles 300 concurrent requests at peak, causing timeouts."},

    {"id": "T-203", "team": "backend", "status": "open",     "priority": "medium",
     "created": date(2025, 11, 1),
     "text": "JWT signature verification fails intermittently — clock skew of 4 seconds between the auth service and API gateway exceeds the token's nbf tolerance."},

    {"id": "T-204", "team": "backend", "status": "resolved", "priority": "medium",
     "created": date(2025, 10, 20),
     "text": "GraphQL resolver N+1 problem — fetching 50 orders triggers 51 separate SQL queries instead of a single join with eager loading."},

    {"id": "T-205", "team": "backend", "status": "open",     "priority": "low",
     "created": date(2025, 11, 7),
     "text": "Celery worker silently dropping tasks when Redis broker restarts — no retry policy or dead-letter queue configured for the affected task type."},

    {"id": "T-206", "team": "backend", "status": "open",     "priority": "high",
     "created": date(2025, 11, 13),
     "text": "Rate limiting not scoping per user — middleware uses a shared Redis key derived from IP, so all users behind a corporate NAT share one limit."},

    {"id": "T-207", "team": "backend", "status": "open",     "priority": "high",
     "created": date(2025, 11, 3),
     "text": "Session cookie persists after logout — token blacklist check is missing from the middleware chain, so invalidated tokens remain usable until natural expiry."},

    {"id": "T-301", "team": "frontend", "status": "open",     "priority": "high",
     "created": date(2025, 11, 6),
     "text": "React hydration mismatch on SSR pages — useLayoutEffect runs server-side in the test environment, producing HTML that diverges from the client tree."},

    {"id": "T-302", "team": "frontend", "status": "open",     "priority": "medium",
     "created": date(2025, 11, 2),
     "text": "Safari 17 CSS grid regression — gap property ignored when the grid parent has overflow: hidden, works correctly in Chrome and Firefox."},

    {"id": "T-303", "team": "frontend", "status": "resolved", "priority": "medium",
     "created": date(2025, 10, 18),
     "text": "Webpack 5 bundle size regression — all date-fns locales included despite tree-shaking config because the barrel export bypasses static analysis."},

    {"id": "T-304", "team": "frontend", "status": "open",     "priority": "low",
     "created": date(2025, 11, 10),
     "text": "WebSocket silently drops after 60 seconds — ALB idle timeout is 60s but client reconnect logic fires at 90s, leaving a 30-second silent failure window."},

    {"id": "T-305", "team": "frontend", "status": "open",     "priority": "low",
     "created": date(2025, 11, 4),
     "text": "Modal dialog fails WCAG 2.1 focus trap — keyboard focus escapes the overlay and reaches background content when tabbing past the last button."},

    {"id": "T-306", "team": "frontend", "status": "resolved", "priority": "low",
     "created": date(2025, 10, 25),
     "text": "Infinite scroll sending duplicate requests on fast scroll — IntersectionObserver fires multiple times before the debounce window closes."},

    {"id": "T-401", "team": "infrastructure", "status": "open",     "priority": "medium",
     "created": date(2025, 11, 11),
     "text": "CI pipeline fails on ARM64 runners — base Docker image has no ARM variant, exec format error at build stage."},

    {"id": "T-402", "team": "infrastructure", "status": "resolved", "priority": "high",
     "created": date(2025, 10, 9),
     "text": "VPN gateway latency spikes at peak hours — BGP route flapping between two peers causing intermittent packet loss across the private subnet."},
]

In [9]:
open_ct     = sum(1 for t in tickets if t["status"] == "open")
resolved_ct = sum(1 for t in tickets if t["status"] == "resolved")
print(f"{len(tickets)} tickets | {open_ct} open | {resolved_ct} resolved")

20 tickets | 14 open | 6 resolved


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts      = [t["text"] for t in tickets]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Shape: {embeddings.shape}  |  norm[0]: {np.linalg.norm(embeddings[0]):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (20, 384)  |  norm[0]: 1.0000


In [11]:
class ContextAwareIndex:
    def __init__(self, embeddings: np.ndarray, documents: list):
        self.embeddings = embeddings   # (N, D), L2-normalized
        self.documents  = documents

    def search(
        self,
        query: str,
        top_k: int       = 5,
        team: str        = None,
        status: str      = None,
        priority: str    = None,
        after:  "date"   = None,
        before: "date"   = None,
        min_score: float = 0.0,
    ) -> list[dict]:

        # Embed the query into the same vector space as the documents
        q_vec = model.encode([query], normalize_embeddings=True)[0]

        # Build a boolean mask — False for any document that fails a filter condition
        mask = np.ones(len(self.documents), dtype=bool)
        for i, doc in enumerate(self.documents):
            if team     and doc["team"]     != team:    mask[i] = False
            if status   and doc["status"]   != status:  mask[i] = False
            if priority and doc["priority"] != priority: mask[i] = False
            if after    and doc["created"]  <  after:   mask[i] = False
            if before   and doc["created"]  >  before:  mask[i] = False

        candidate_idx = np.where(mask)[0]
        if len(candidate_idx) == 0:
            return []

        # Score only the candidates that passed the filter
        scores = self.embeddings[candidate_idx] @ q_vec

        # Drop anything below the minimum score threshold, sort, return top-k
        valid = np.where(scores >= min_score)[0]
        if len(valid) == 0:
            return []

        top_local  = np.argsort(scores[valid])[::-1][:top_k]
        top_global = candidate_idx[valid[top_local]]

        return [
            {**self.documents[i], "score": float(scores[valid[top_local[j]]])}
            for j, i in enumerate(top_global)
        ]


index = ContextAwareIndex(embeddings, tickets)

In [12]:
def show(label, results):
    print(f"\nQuery: {label}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['id']}  {r['team']:>14}  {r['status']:>8}"
              f"  {r['priority']:>6}  {r['created']}")
        print(f"           {r['text'][:85]}...")

In [13]:
results = index.search("authentication token expiry and session management", top_k=4)
show("'authentication token expiry and session management'  (no filters)", results)


Query: 'authentication token expiry and session management'  (no filters)
  [0.6133]  T-207         backend      open    high  2025-11-03
           Session cookie persists after logout — token blacklist check is missing from the midd...
  [0.4958]  T-201         backend      open    high  2025-11-05
           OAuth2 token refresh fails intermittently — race condition in the token cache where t...
  [0.3459]  T-203         backend      open  medium  2025-11-01
           JWT signature verification fails intermittently — clock skew of 4 seconds between the...
  [0.1714]  T-206         backend      open    high  2025-11-13
           Rate limiting not scoping per user — middleware uses a shared Redis key derived from ...


In [22]:
results = index.search(
    "authentication token expiry and session management",
    top_k=4,
    status="open",
    before=date(2025, 11, 10),
)
show("same query  [status=open, before=2025-11-10]", results)


Query: same query  [status=open, before=2025-11-10]
  [0.6133]  T-207         backend      open    high  2025-11-03
           Session cookie persists after logout — token blacklist check is missing from the midd...
  [0.4958]  T-201         backend      open    high  2025-11-05
           OAuth2 token refresh fails intermittently — race condition in the token cache where t...
  [0.3459]  T-203         backend      open  medium  2025-11-01
           JWT signature verification fails intermittently — clock skew of 4 seconds between the...
  [0.1419]  T-202         backend      open    high  2025-11-09
           Database connection pool exhausted under load — pool capped at 20 connections but the...


In [23]:
results = index.search(
    "resource exhaustion and memory pressure under load",
    top_k=2,
    status="open",
    priority="high",
)
show("'resource exhaustion and memory pressure'  [status=open, priority=high]", results)


Query: 'resource exhaustion and memory pressure'  [status=open, priority=high]
  [0.3877]  T-202         backend      open    high  2025-11-09
           Database connection pool exhausted under load — pool capped at 20 connections but the...
  [0.2908]  T-101  infrastructure      open    high  2025-11-03
           Kubernetes pod keeps crashing with OOMKilled — memory limits on the ML inference cont...


In [24]:
import json

# Write the embedding matrix and ticket metadata to disk
np.save("ticket_embeddings.npy", embeddings)

with open("ticket_metadata.json", "w") as f:
    json.dump(
        [{**t, "created": t["created"].isoformat()} for t in tickets],
        f, indent=2,
    )

print("Saved.")

Saved.


In [25]:
from datetime import date
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# Restore the embedding matrix, deserialize the metadata, rebuild the index
embeddings_loaded = np.load("ticket_embeddings.npy")

with open("ticket_metadata.json") as f:
    tickets_loaded = json.load(f)
for t in tickets_loaded:
    t["created"] = date.fromisoformat(t["created"])

model = SentenceTransformer("all-MiniLM-L6-v2")
index = ContextAwareIndex(embeddings_loaded, tickets_loaded)

print(f"Reloaded: {embeddings_loaded.shape[0]} docs, {embeddings_loaded.shape[1]}D.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reloaded: 20 docs, 384D.
